In [9]:
"""
TRAIN MODELS
======================
Trains regression models on voter profile + country state → ideal vector.

INPUT  (25 columns): voter demographics + attitudes + 5 country fields
OUTPUT (9 columns):  voter's preference profile

The model learns:
  "given this voter AND this country context, predict their ideal platform"
"""
import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
# import matplotlib.pyplot as plt
from pathlib import Path

import joblib
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import (
    explained_variance_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.multioutput import MultiOutputRegressor, RegressorChain
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

BASE_DIR = Path().resolve()
Path("models").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)
DATA_PATH = BASE_DIR / "data" / "data" / "non-linear-voters.csv"

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f"\n{len(df)} rows loaded")
# df.info()
y_cols = [
    "understand_government_institutions",
    "believe_government_institutions",
    "every_person_is_expert",
    "probability_take_part",
    "candidate_positive_importance",
    "candidate_negative_fair_importance",
    "like_easy_decision",
    "like_strong_leader_over_law",
    "person_or_government_importance",
]
X = df.drop(columns=y_cols)
X = X.drop(columns=["voter_id", "snapshot_id", "base_voter_id", "source"])

y = df[y_cols]
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "string"]).columns
groups = df["base_voter_id"]

# print(num_cols)
# print(cat_cols)
ORDINAL_COLS = [
    "age",
    "education",
    "economic_status",
    "finance_independent",
    "understanding_econ",
    "understanding_safety",
    "understanding_social",
]
NOMINAL_COLS = ["sex", "nationality", "identity"]
AGE_GROUPS = ["18-30", "31-45", "45-60", "60+"]
EDUCATIONS_LIST = ["9-11 класи", "Профтех/коледж", "Незакінчена вища", "Вища освіта"]
ECON_STATUS_LIST = [
    "Низький",
    "Нижче середнього",
    "Середній",
    "Вище середнього",
    "Високий",
    "Заможний",
]
FIN_INDEPENDENT_LIST = [
    "Повністю залежний",
    "Переважно залежний",
    "Частково і так, і так",
    "Переважно самостійний",
    "Повністю самостійний",
]
UNDERSTANDING = [
    "Не орієнтується у темі та не може пояснити базові поняття",
    "Розуміє загальні ідеї, але складно пояснювати або застосовувати їх",
    "Може пояснити основні поняття іншій людині та розуміє типові приклади",
    "Впевнено застосовує знання на практиці та може аналізувати ситуації",
    "Може критично оцінювати інформацію, знаходити помилки та аргументувати позицію",
]
categories = [
    AGE_GROUPS,
    EDUCATIONS_LIST,
    ECON_STATUS_LIST,
    FIN_INDEPENDENT_LIST,
    UNDERSTANDING,
    UNDERSTANDING,
    UNDERSTANDING,
]

ordinal_transformer = Pipeline(
    [
        (
            "encode",
            OrdinalEncoder(
                categories=categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

nominal_transformer = Pipeline(
    [
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# print(ordinal_transformer.named_steps)
# sample = pd.DataFrame({
#     "age": ["60+"],
#     "education": ["Вища освіта"],
#     "economic_status": ["Високий"],
#     "finance_independent": ["Повністю самостійний"],
#     "understanding_econ":["Може пояснити основні поняття іншій людині та розуміє типові приклади"],
#     "understanding_safety":["Не орієнтується у темі та не може пояснити базові поняття"],
#     "understanding_social":["Може критично оцінювати інформацію, знаходити помилки та аргументувати позицію"]
# })

# print(ordinal_transformer.fit(X[ORDINAL_COLS]).transform(sample))

# nominal_transformer.fit(X[NOMINAL_COLS])
# print(
#     nominal_transformer.named_steps["encode"].categories_
# )

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("ordinal_transformer", ordinal_transformer, ORDINAL_COLS),
        ("nominal_transformer", nominal_transformer, NOMINAL_COLS),
    ]
)

# names = preprocessor.get_feature_names_out()

# for i, name in enumerate(names):
#     print(i, name)

gss = GroupShuffleSplit(n_splits=3, test_size=0.2, random_state=42)


def make_model(model, use_chain=False):
    wrapper = RegressorChain(model) if use_chain else MultiOutputRegressor(model)
    return Pipeline(
        [
            ("preprocess", preprocessor),
            ("model", wrapper),
        ]
    )


MODELS = {
    "Multi_RandomForestRegressor": make_model(
        RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
        use_chain=False,
    ),
    "Chain_RandomForestRegressor": make_model(
        RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
        use_chain=True,
    ),
    "Multi_HistGradientBoostingRegressor": make_model(
        HistGradientBoostingRegressor(
            random_state=42, max_iter=100, learning_rate=0.05, max_depth=5
        ),
        use_chain=False,
    ),
    "Chain_HistGradientBoostingRegressor": make_model(
        HistGradientBoostingRegressor(
            random_state=42, max_iter=100, learning_rate=0.05, max_depth=5
        ),
        use_chain=True,
    ),
}

results = []

for name, model in MODELS.items():
    print(f"\nНавчання моделі: {name}...")

    fold_scores = []

    for train_idx, test_idx in gss.split(X, y, groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipe = clone(model)
        pipe.fit(X_train, y_train)
        Y_pred = pipe.predict(X_test)

        fold_scores.append(
            {
                "mae": mean_absolute_error(y_test, Y_pred),
                "r2": r2_score(y_test, Y_pred),
                "mse": mean_squared_error(y_test, Y_pred),
                "evs": explained_variance_score(y_test, Y_pred),
            }
        )

    test_mae = np.mean([f["mae"] for f in fold_scores])
    test_r2 = np.mean([f["r2"] for f in fold_scores])
    mse = np.mean([f["mse"] for f in fold_scores])
    evs = np.mean([f["evs"] for f in fold_scores])
    print("MAE std:", np.std([f["mae"] for f in fold_scores]))
    print("R2 std:", np.std([f["r2"] for f in fold_scores]))

    results.append(
        {
            "model": name,
            "test_mae": round(test_mae, 3),
            "test_r2": round(test_r2, 3),
            "mse": round(mse, 3),
            "evs": round(evs, 3),
        }
    )

    print(f"  MAE = {test_mae:.3f}")
    print(f"  R^2  = {test_r2:.3f}")
    print(f"  MSE = {mse:.3f}")
    print(f"  EVS  = {evs:.3f}")


res_df = pd.DataFrame(results)
print("\n=== Фінальні результати ===")
print(res_df.to_string(index=False))

print("\nЗбереження моделі")

best_model_name = min(results, key=lambda r: r["test_mae"])["model"]

best_pipe = None

for name, model in MODELS.items():
    print(f"Фінальне тренування: {name}")

    model.fit(X, y)

    joblib.dump(model, f"models/updated_{name}.pkl")

    if name == best_model_name:
        best_pipe = model
        joblib.dump(best_pipe, "models/update_best_model.pkl")

with open("models/updated_best_model.txt", "w") as f:
    f.write(best_model_name)

res_df.to_csv("results/updated_model_comparison.csv", index=False)

print(f"Збережено найкращу модель: {best_model_name}")


20074 rows loaded

Навчання моделі: Multi_RandomForestRegressor...
MAE std: 0.0013887490789613317
R2 std: 0.0012192770027286569
  MAE = 1.593
  R^2  = 0.503
  MSE = 7.706
  EVS  = 0.503

Навчання моделі: Chain_RandomForestRegressor...
MAE std: 0.00145121764681595
R2 std: 0.0013580468065485973
  MAE = 1.591
  R^2  = 0.505
  MSE = 7.825
  EVS  = 0.505

Навчання моделі: Multi_HistGradientBoostingRegressor...
MAE std: 0.002540290275092717
R2 std: 0.0008314965268494401
  MAE = 1.568
  R^2  = 0.516
  MSE = 7.364
  EVS  = 0.516

Навчання моделі: Chain_HistGradientBoostingRegressor...
MAE std: 0.0027170564321770713
R2 std: 0.0012045161847796693
  MAE = 1.565
  R^2  = 0.517
  MSE = 7.447
  EVS  = 0.517

=== Фінальні результати ===
                              model  test_mae  test_r2   mse   evs
        Multi_RandomForestRegressor     1.593    0.503 7.706 0.503
        Chain_RandomForestRegressor     1.591    0.505 7.825 0.505
Multi_HistGradientBoostingRegressor     1.568    0.516 7.364 0.516